In [ ]:
# -------------------------------
# Step 1. Imports
# -------------------------------
import os, glob, random, shutil, urllib.request, zipfile
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms

In [ ]:
# -------------------------------
# Step 2. Setup paths
# -------------------------------

from google.colab import drive
drive.mount('/content/drive')
drive_dir = '/content/drive/MyDrive/'

Mounted at /content/drive


In [ ]:
def mlm_margin_top1_top2_at_first_mask(outputs, input_ids, mask_token_id):
    # outputs.logits: (B, T, V)
    logits = outputs.logits
    B, T, V = logits.shape
    device = logits.device

    mask_pos = (input_ids == mask_token_id).float().argmax(dim=1)  # first mask per example
    sel = logits[torch.arange(B, device=device), mask_pos, :]      # (B, V)

    top2 = torch.topk(sel, k=2, dim=-1).values
    return top2[:, 0] - top2[:, 1]  # (B,)

def cls_margin_top1_top2(outputs):
    # outputs.logits: (B, K)
    logits = outputs.logits
    top2 = torch.topk(logits, k=2, dim=-1).values
    return top2[:, 0] - top2[:, 1]

In [ ]:
import re
import torch
import torch.nn as nn
from collections import OrderedDict

# -----------------------------
# Select layers: Q/K/V + all Linear + all LayerNorm (with bias)
# -----------------------------
def get_layers_qkv_ln_linear(model: nn.Module, name_regex: str | None = None):
    """
    Returns [(name, module), ...] for:
      - nn.Linear with bias (includes BERT Q/K/V)
      - nn.LayerNorm with bias (beta)
    Optional regex filters layer names (regex search).
    """
    pat = re.compile(name_regex) if name_regex else None
    layers = []
    for name, m in model.named_modules():
        ok = (isinstance(m, nn.Linear) and m.bias is not None) or \
             (isinstance(m, nn.LayerNorm) and getattr(m, "bias", None) is not None)
        if not ok:
            continue
        if pat and not pat.search(name):
            continue
        layers.append((name, m))
    return layers

def _broadcast_bias_like(module: nn.Module, u: torch.Tensor) -> torch.Tensor:
    """
    Broadcast the module's additive bias to match u's shape.
    Linear: bias (C,) -> (..., C)
    LayerNorm: bias (hidden,) or normalized_shape -> (..., normalized_shape)
    """
    if isinstance(module, nn.Linear):
        b = module.bias
        view = [1] * (u.ndim - 1) + [-1]
        return b.view(*view).expand_as(u)

    if isinstance(module, nn.LayerNorm):
        b = module.bias
        view = [1] * (u.ndim - b.ndim) + list(b.shape)
        return b.view(*view).expand_as(u)

    return torch.zeros_like(u)

# -----------------------------
# Layerwise BDI via gradient-weighted contributions
# -----------------------------
@torch.enable_grad()
def layerwise_bdi_qkv_ln_linear(
    model: nn.Module,
    model_inputs: dict,
    margin_fn,                        # callable(outputs) -> tensor (B,) or scalar
    layers: list[tuple[str, nn.Module]],
    eps: float = 1e-12,
):
    """
    For each layer l:
      C_feat^l = sum(g * (u - u_bias))
      C_bias^l = sum(g * u_bias)
      BDI^l    = |C_bias^l| / (|C_feat^l| + |C_bias^l| + eps)

    NOTE: For internal layers this is a *local (gradient-linearized) estimate*.
    It is exact only at the final affine head.
    """
    model.eval()
    store = {}
    hooks = []

    def make_hook(name):
        def hook(module, inputs, output):
            if not torch.is_tensor(output):
                return
            u = output
            u.retain_grad()
            ub = _broadcast_bias_like(module, u)
            store[name] = {"u": u, "ub": ub, "type": module.__class__.__name__, "shape": tuple(u.shape)}
        return hook

    for name, m in layers:
        hooks.append(m.register_forward_hook(make_hook(name)))

    outputs = model(**model_inputs)

    margin = margin_fn(outputs)
    if torch.is_tensor(margin) and margin.ndim > 0:
        margin = margin.sum()

    model.zero_grad(set_to_none=True)
    margin.backward()

    out = OrderedDict()
    for name, _ in layers:
        if name not in store:
            continue
        u  = store[name]["u"]
        ub = store[name]["ub"]
        g  = u.grad
        if g is None:
            continue

        uf = u - ub
        red = tuple(range(1, u.ndim)) if u.ndim >= 2 else ()

        C_feat = (g * uf).sum(dim=red)
        C_bias = (g * ub).sum(dim=red)
        BDI    = C_bias.abs() / (C_feat.abs() + C_bias.abs() + eps)

        out[name] = {
            "type":  store[name]["type"],
            "shape": store[name]["shape"],
            "C_feat": C_feat.detach().cpu(),
            "C_bias": C_bias.detach().cpu(),
            "BDI":    BDI.detach().cpu(),
        }

    for h in hooks:
        h.remove()

    return out

In [ ]:
def mlm_margin_top1_top2_first_mask(outputs, input_ids, mask_token_id):
    logits = outputs.logits                      # (B, T, V)
    B, T, V = logits.shape
    device = logits.device

    # first [MASK] per example
    mask_pos = (input_ids == mask_token_id).float().argmax(dim=1)  # (B,)
    sel = logits[torch.arange(B, device=device), mask_pos, :]      # (B, V)

    top2 = torch.topk(sel, k=2, dim=-1).values
    return top2[:, 0] - top2[:, 1]

In [ ]:
# ============================================================
# BERT MLM: Complex prompts → greedy mask fill + layerwise BDI
# Includes Q/K/V bias (Linear), LayerNorm beta, and all Linear biases
# ============================================================

import re
import torch
import torch.nn as nn
import pandas as pd
from collections import OrderedDict
from transformers import AutoTokenizer, AutoModelForMaskedLM



# -----------------------------
# 1) Layer selection: Linear (incl. Q/K/V) + LayerNorm (beta)
# -----------------------------
def get_layers_qkv_ln_linear(model: nn.Module, name_regex: str | None = None):
    pat = re.compile(name_regex) if name_regex else None
    layers = []
    for name, m in model.named_modules():
        ok = (isinstance(m, nn.Linear) and m.bias is not None) or \
             (isinstance(m, nn.LayerNorm) and getattr(m, "bias", None) is not None)
        if not ok:
            continue
        if pat and not pat.search(name):
            continue
        layers.append((name, m))
    return layers

def _broadcast_bias_like(module: nn.Module, u: torch.Tensor) -> torch.Tensor:
    if isinstance(module, nn.Linear):
        b = module.bias
        view = [1] * (u.ndim - 1) + [-1]  # (..., C)
        return b.view(*view).expand_as(u)
    if isinstance(module, nn.LayerNorm):
        b = module.bias
        view = [1] * (u.ndim - b.ndim) + list(b.shape)
        return b.view(*view).expand_as(u)
    return torch.zeros_like(u)

# -----------------------------
# 2) Layerwise BDI (gradient-weighted; exact only at affine head)
# -----------------------------
@torch.enable_grad()
def layerwise_bdi_qkv_ln_linear(
    model: nn.Module,
    model_inputs: dict,
    margin_fn,                       # callable(outputs)->(B,) or scalar
    layers: list[tuple[str, nn.Module]],
    eps: float = 1e-12
):
    model.eval()
    store = {}
    hooks = []

    def make_hook(name):
        def hook(module, inputs, output):
            if not torch.is_tensor(output):
                return
            u = output
            u.retain_grad()
            ub = _broadcast_bias_like(module, u)
            store[name] = {"u": u, "ub": ub, "type": module.__class__.__name__}
        return hook

    for name, m in layers:
        hooks.append(m.register_forward_hook(make_hook(name)))

    outputs = model(**model_inputs)

    margin = margin_fn(outputs)
    if torch.is_tensor(margin) and margin.ndim > 0:
        margin = margin.sum()

    model.zero_grad(set_to_none=True)
    margin.backward()

    out = OrderedDict()
    for name, _ in layers:
        if name not in store:
            continue
        u = store[name]["u"]
        ub = store[name]["ub"]
        g = u.grad
        if g is None:
            continue

        uf = u - ub
        red = tuple(range(1, u.ndim)) if u.ndim >= 2 else ()
        C_feat = (g * uf).sum(dim=red)
        C_bias = (g * ub).sum(dim=red)
        BDI = C_bias.abs() / (C_feat.abs() + C_bias.abs() + eps)

        out[name] = {
            "type": store[name]["type"],
            "C_feat": C_feat.detach().cpu(),
            "C_bias": C_bias.detach().cpu(),
            "BDI": BDI.detach().cpu(),
        }

    for h in hooks:
        h.remove()

    return out

# -----------------------------
# 3) MLM margin: top1 - top2 at first [MASK]
# -----------------------------
def mlm_margin_top1_top2_first_mask(outputs, input_ids, mask_token_id):
    logits = outputs.logits  # (B, T, V)
    B, T, V = logits.shape
    device = logits.device
    mask_pos = (input_ids == mask_token_id).float().argmax(dim=1)  # first [MASK]
    sel = logits[torch.arange(B, device=device), mask_pos, :]      # (B, V)
    top2 = torch.topk(sel, k=2, dim=-1).values
    return top2[:, 0] - top2[:, 1]

# -----------------------------
# 4) Greedy fill first mask (completion)
# -----------------------------
@torch.no_grad()
def fill_first_mask_greedy(model, tok, enc):
    out = model(**enc)
    logits = out.logits  # (1, T, V)
    input_ids = enc["input_ids"][0]
    mask_pos = (input_ids == tok.mask_token_id).nonzero(as_tuple=False).squeeze(-1)
    if mask_pos.numel() == 0:
        return None
    pos = int(mask_pos[0].item())

    probs = torch.softmax(logits[0, pos], dim=-1)
    top2p = torch.topk(probs, k=2)
    top1_id = int(top2p.indices[0].item())
    top2_id = int(top2p.indices[1].item())
    top1_prob = float(top2p.values[0].item())

    top2_logits = torch.topk(logits[0, pos], k=2).values
    margin = float((top2_logits[0] - top2_logits[1]).item())

    filled = enc["input_ids"].clone()
    filled[0, pos] = top1_id
    completed_text = tok.decode(filled[0], skip_special_tokens=True)

    return {
        "mask_pos": pos,
        "completed_text": completed_text,
        "top1_id": top1_id,
        "top2_id": top2_id,
        "top1_token": tok.convert_ids_to_tokens(top1_id),
        "top2_token": tok.convert_ids_to_tokens(top2_id),
        "top1_prob": top1_prob,
        "margin": margin,
    }

# -----------------------------
# 5) Run prompts: completion + top-K BDI layers; save CSV
# -----------------------------
def run_complex_prompts(model, tok, device, layers, complex_prompts, topk_layers=5, max_prompts=None, to_csv=None):
    rows = []
    n = len(complex_prompts) if max_prompts is None else min(len(complex_prompts), max_prompts)

    for idx in range(n):
        prompt = complex_prompts[idx]
        enc = tok(prompt, return_tensors="pt").to(device)

        comp = fill_first_mask_greedy(model, tok, enc)
        if comp is None:
            rows.append({"idx": idx, "prompt": prompt, "error": "no_mask"})
            continue

        res = layerwise_bdi_qkv_ln_linear(
            model=model,
            model_inputs=enc,
            margin_fn=lambda out, _enc=enc: mlm_margin_top1_top2_first_mask(out, _enc["input_ids"], tok.mask_token_id),
            layers=layers
        )

        items = [(name, float(v["BDI"][0]), float(v["C_feat"][0]), float(v["C_bias"][0]), v["type"])
                 for name, v in res.items()]
        items.sort(key=lambda x: x[1], reverse=True)
        top_items = items[:topk_layers]
        best_name, best_bdi, best_cfeat, best_cbias, best_typ = top_items[0]

        print("\n" + "=" * 110)
        print(f"[{idx}] PROMPT:    {prompt}")
        print(f"     COMPLETED: {comp['completed_text']}")
        print(f"     MASK FILL: {comp['top1_token']} (p={comp['top1_prob']:.4f}) vs {comp['top2_token']} | margin={comp['margin']:.4f}")
        print(f"     Top {topk_layers} layers by BDI:")
        for name, bdi, cfeat, cbias, typ in top_items:
            print(f"       {bdi:7.4f}  {typ:10s}  C_feat={cfeat:+.4f}  C_bias={cbias:+.4f}  {name}")

        rows.append({
            "idx": idx,
            "prompt": prompt,
            "completed_text": comp["completed_text"],
            "mask_pos": comp["mask_pos"],
            "top1_token": comp["top1_token"],
            "top2_token": comp["top2_token"],
            "top1_prob": comp["top1_prob"],
            "margin": comp["margin"],
            "best_layer": best_name,
            "best_layer_type": best_typ,
            "best_bdi": best_bdi,
            "best_C_feat": best_cfeat,
            "best_C_bias": best_cbias,
        })

    df = pd.DataFrame(rows)
    if to_csv:
        df.to_csv(to_csv, index=False)
        print(f"\nSaved to: {to_csv}")
    return df

# -----------------------------
# 6) Instantiate model/tokenizer + run
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForMaskedLM.from_pretrained(ckpt).to(device)

# Track encoder + MLM head + embeddings (keeps Q/K/V + LN inside encoder)
name_regex = r"^(bert\.encoder\.layer\.\d+|cls\.predictions|bert\.embeddings)\."
layers = get_layers_qkv_ln_linear(model, name_regex=name_regex)

df_results = run_complex_prompts(
    model=model,
    tok=tok,
    device=device,
    layers=layers,
    complex_prompts=complex_prompts,
    topk_layers=5,
    max_prompts=None,
    to_csv="bert_mlm_complex_prompts_bdi_complex_prompts.csv"
)

df_results.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[0] PROMPT:    She reported months of polyuria and polydipsia; HbA1c returned at 11.2%, and the clinician suspected [MASK].
     COMPLETED: she reported months of polyuria and polydipsia ; hba1c returned at 11. 2 %, and the clinician suspected infection.
     MASK FILL: infection (p=0.0812) vs cancer | margin=0.2378
     Top 5 layers by BDI:
        0.8560  Linear      C_feat=-0.0346  C_bias=+0.2058  bert.encoder.layer.7.attention.self.query
        0.8390  Linear      C_feat=-0.0903  C_bias=-0.4710  bert.encoder.layer.6.intermediate.dense
        0.8390  LayerNorm   C_feat=+0.1069  C_bias=+0.5569  bert.encoder.layer.10.attention.output.LayerNorm
        0.8337  Linear      C_feat=+0.0378  C_bias=-0.1893  bert.encoder.layer.8.attention.self.value
        0.7404  Linear      C_feat=+0.0197  C_bias=-0.0561  bert.encoder.layer.3.attention.self.query

[1] PROMPT:    On arrival the patient was febrile and tachycardic; blood cultures later grew gram-positive cocci, consistent with [MASK].
 

,idx,prompt,completed_text,mask_pos,top1_token,top2_token,top1_prob,margin,best_layer,best_layer_type,best_bdi,best_C_feat,best_C_bias
0,0,She reported months of polyuria and polydipsia...,she reported months of polyuria and polydipsia...,29,infection,cancer,0.081226,0.237790,bert.encoder.layer.7.attention.self.query,Linear,0.855998,-0.034628,0.205844
1,1,On arrival the patient was febrile and tachyca...,on arrival the patient was febrile and tachyca...,27,symptoms,this,0.236629,1.685245,bert.encoder.layer.4.output.dense,Linear,0.997119,0.000887,-0.306992
2,2,"The ECG showed ST elevation in leads II, III, ...","the ecg showed st elevation in leads ii, iii, ...",23,phase,condition,0.079694,0.034723,bert.encoder.layer.1.attention.output.dense,Linear,0.914494,-0.016554,0.177046
3,3,CT angiography demonstrated a filling defect i...,ct angiography demonstrated a filling defect i...,17,age,mri,0.080240,0.310864,bert.encoder.layer.6.attention.output.dense,Linear,0.922861,-0.045265,-0.541532
4,4,The chest radiograph showed a lobar consolidat...,the chest radiograph showed a lobar consolidat...,20,pneumonia,health,0.161443,0.771935,bert.encoder.layer.2.attention.output.dense,Linear,0.984143,0.004470,-0.277436
